In [0]:
# Install select libraries that depend on databricks sdk >= 0.65.0

# Install a coherent 0.3.x stack (compatible with unitycatalog-langchain and databricks-langchain)
%pip install -U \
  "langchain<0.4,>=0.3.27" \
  "langchain-core<0.4,>=0.3.79" \
  "langchain-community<0.4,>=0.2" \
  "langchain-text-splitters<1.0,>=0.3.9" \
  "langchain-openai<0.3,>=0.2.0" \
  "pydantic>=2.0.0,<3.0.0" \
  "databricks-sdk>=0.65.0" \
  "databricks-langchain==0.8.2"

# Restart the Python VM so the environment picks up the new packages
%restart_python

In [0]:
%run ../../Includes/_common

In [0]:
# Create a python DA object from the dbacademy.ops.meta table
DA = DBAcademyHelper()
DA.init()

In [0]:
def use_uc_env():
    catalog_name = DA.catalog_name
    schema_name = DA.schema_name 

    spark.sql(f"USE CATALOG {DA.catalog_name}")
    spark.sql(f"USE SCHEMA {DA.schema_name}")
    return catalog_name, schema_name 

In [0]:
def process_airbnb_dataset(databricks_share_name: str):
    # Read the CSV file from the volume with headers
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .option("multiLine", "true") \
        .option("escape", '"') \
        .load(f"/Volumes/{databricks_share_name}/v01/sf-listings/sf-airbnb.csv")

    # Write as a Delta table
    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("sf_airbnb_listings")
        
    print(f"✅ Successfully created table {DA.catalog_name}.{DA.schema_name}.sf_airbnb_listings.")

In [0]:
# Set UC environment
catalog_name, schema_name = use_uc_env()

# Process the Airbnb dataset
process_airbnb_dataset(
        databricks_share_name = "dbacademy_airbnb"
        )

In [0]:
def set_environment_and_tools(catalog_name:str, schema_name:str) -> None:
    """
    Sets the environment and tools for the notebook.
    """

    catalog_query = f"USE CATALOG {catalog_name}"
    schema_query = f"USE SCHEMA {schema_name}"
    tool1_name = "avg_fare_by_zip"
    tool2_name = "pickup_zip_code"
    tool1 = f"""
            -- Create Tool 1 that gets the average fare by pickup zip code
            CREATE OR REPLACE FUNCTION {tool1_name}(
            pickup_zip_code INT COMMENT "The pickup ZIP code to filter by"
            )
            RETURNS DOUBLE
            LANGUAGE SQL
            DETERMINISTIC
            COMMENT 'Calculates the average fare amount for trips from a specific pickup ZIP code. Returns the average fare as a numeric value.'
            RETURN 
            SELECT AVG(fare_amount)
            FROM samples.nyctaxi.trips
            WHERE pickup_zip = pickup_zip_code
            AND fare_amount IS NOT NULL;
    """

    tool2 = f"""
        -- Create Tool 2 that counts the number of long distance trips
        CREATE OR REPLACE FUNCTION cnt_lng_dist_trip(
        {tool2_name} INT COMMENT "The pickup ZIP code to filter by",
        min_distance DOUBLE COMMENT "The minimum trip distance in miles"
        )
        RETURNS BIGINT
        LANGUAGE SQL
        DETERMINISTIC
        COMMENT 'Counts the number of trips longer than the specified distance from a given pickup ZIP code. Returns the count as an integer.'
        RETURN
        SELECT COUNT(*)
        FROM samples.nyctaxi.trips
        WHERE pickup_zip = pickup_zip_code
            AND trip_distance > min_distance;
    """
    spark.sql(catalog_query)
    spark.sql(schema_query)
    spark.sql(tool1).collect()
    spark.sql(tool2).collect()
    print(f"Created function(s): {tool1_name} and {tool2_name}")
    return None

In [0]:
set_environment_and_tools(catalog_name, schema_name)